In [5]:
from itertools import combinations
import os
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import networkx as nx

In [2]:
profiles = pd.read_csv('profiles.csv')

In [3]:
out_dir = "full_graph_parquet"
os.makedirs(out_dir, exist_ok=True)

# 1. Сохраняем вершины
nodes_df = profiles[
    [
        "profile_id",
        "entity_id_last_notna",
        "first_name_last_notna",
        "last_name_last_notna",
        "email_last_notna",
        "phone_last_notna",
    ]
].rename(columns={
    "entity_id_last_notna": "entity_id",
    "first_name_last_notna": "first_name",
    "last_name_last_notna": "last_name",
    "email_last_notna": "email",
    "phone_last_notna": "phone",
})

nodes_df.to_parquet(f"{out_dir}/nodes.parquet", index=False)

# 2. Подготовка
ids = profiles["profile_id"].tolist()
entity_map = dict(zip(profiles["profile_id"], profiles["entity_id_last_notna"]))

batch = []
batch_size = 1_000_000
writer = None
edges_path = f"{out_dir}/edges.parquet"

# 3. Генерация всех пар
for u, v in combinations(ids, 2):
    same_ent = int(entity_map[u] == entity_map[v])
    batch.append((u, v, 1, same_ent))

    if len(batch) >= batch_size:
        df_batch = pd.DataFrame(
            batch,
            columns=["u", "v", "weight", "label_same_entity"]
        )
        table = pa.Table.from_pandas(df_batch, preserve_index=False)

        if writer is None:
            writer = pq.ParquetWriter(edges_path, table.schema, compression="snappy")

        writer.write_table(table)
        batch.clear()

# 4. Хвост
if batch:
    df_batch = pd.DataFrame(
        batch,
        columns=["u", "v", "weight", "label_same_entity"]
    )
    table = pa.Table.from_pandas(df_batch, preserve_index=False)

    if writer is None:
        writer = pq.ParquetWriter(edges_path, table.schema, compression="snappy")

    writer.write_table(table)

if writer is not None:
    writer.close()

print("Done.")
print("Nodes saved to:", f"{out_dir}/nodes.parquet")
print("Edges saved to:", edges_path)
print("Num nodes:", len(ids))
print("Expected num edges:", len(ids) * (len(ids) - 1) // 2)

Done.
Nodes saved to: full_graph_parquet/nodes.parquet
Edges saved to: full_graph_parquet/edges.parquet
Num nodes: 61927
Expected num edges: 1917445701


In [8]:
nodes_df = pd.read_parquet("full_graph_parquet/nodes.parquet")
print(nodes_df["entity_id"].value_counts().head(20))

entity_id
46167d1f0d191d82ea20c4c26da86d0cda67a1256ad453d8563d27995448f040    116
17bb90b8fadb73e27adc5f9c44de6189933873d0671ef938ffaa4a1582de072d     37
8c46ac60a1d0186d5e3c23f4de305aa472fa741ff3b30f62409eeebe64ee78d4     22
4e5fe5b3d106905f61229dc7332eaf738dd17a14cd3ee7c4289efa7cfc834ceb     10
c842f8250569d574da72c26ef4b568d688aed68f83544b7c10e24cfba5957470      7
b2493335d32f352fc7ec47c571a07324ecfcd40eacd49092322ece696633f804      6
94c8f3e4eded528564dfc29171de2d5c7e26ed727514afd4bdfe650cdda830dd      6
95aa0ddff94d2d45b94731bc389ec230b6ca3d60b9b5a4b9910552896f40a3a0      6
d8b4d482e0af412f803cdb21dc3e45be33945b85c3311ebda7b72c58373ca67d      5
8ce5e66846ae80643ec6b46dbfc1231192904b9350f4359630f2bef1346ae212      5
3b16d5446c807f30bdc8cbaffebf4be6ce7f2969584afabf77fb2203d503f867      5
42247ceb30761e7ee3094b626728ebb5cfa2877305915cd5ff06914414e9aa6c      5
0053d4a01d567b83c8ecb9111a37e8c57b18fe4fedff15ecf25fdc1b331fa753      4
e10269a052ad9356ad6336621079c5050e0bf37c12e0d9519163ce

In [20]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from catboost import CatBoostClassifier

In [22]:
# 1. Загружаем nodes
profiles = pd.read_parquet(f"{out_dir}/nodes.parquet").copy()

for col in ["email", "phone", "first_name", "last_name"]:
    if col in profiles.columns:
        profiles[col] = profiles[col].astype("string").fillna("")

# если email_domain / phone_prefix нет в nodes.parquet — достроим
if "email_domain" not in profiles.columns:
    profiles["email_domain"] = profiles["email"].str.extract(r"@(.+)$")[0].fillna("").astype("string")
if "phone_prefix" not in profiles.columns:
    profiles["phone_prefix"] = profiles["phone"].astype("string").fillna("").str.slice(0, 3)

# индекс по profile_id
profiles = profiles.set_index("profile_id", drop=False)

In [25]:
import numpy as np
from sklearn.model_selection import train_test_split

# 2. Сплит по entity_id
# Приводим к numpy-массиву простых python-строк
unique_entities = (
    profiles["entity_id"]
    .dropna()
    .astype("string")     # на всякий случай
    .unique()
)

# ВАЖНО: превратить в numpy array / list
unique_entities = np.array(unique_entities, dtype=object)  # или list(...)

train_entities, test_entities = train_test_split(
    unique_entities, test_size=0.2, random_state=42
)
train_entities, valid_entities = train_test_split(
    train_entities, test_size=0.2, random_state=42
)

entity_split = {}
for e in train_entities:
    entity_split[e] = "train"
for e in valid_entities:
    entity_split[e] = "valid"
for e in test_entities:
    entity_split[e] = "test"

profiles["split"] = profiles["entity_id"].map(entity_split)
# быстрые мапы
entity_map = profiles["entity_id"].to_dict()
split_map = profiles["split"].to_dict()

In [26]:
# 3. Вспомогательные функции
def safe_str_eq(a, b):
    return (a.fillna("") == b.fillna("")).astype("int8")

def safe_num_diff(a, b):
    a = pd.to_numeric(a, errors="coerce")
    b = pd.to_numeric(b, errors="coerce")
    return (a - b).abs().fillna(-1)

def parse_setlike(x):
    if isinstance(x, list):
        return set(x)
    if pd.isna(x):
        return set()
    if isinstance(x, str):
        # если колонка сохранена строкой вида "['a', 'b']", можно позже улучшить parser
        return set([x]) if x else set()
    return set()

def jaccard_series(col_u, col_v):
    vals = []
    for a, b in zip(col_u, col_v):
        sa, sb = parse_setlike(a), parse_setlike(b)
        if not sa and not sb:
            vals.append(0.0)
        else:
            vals.append(len(sa & sb) / max(1, len(sa | sb)))
    return pd.Series(vals, dtype="float32")

In [27]:
# -------------------------
# 4. Читаем edges.parquet и собираем обучающую выборку
# -------------------------
pf = pq.ParquetFile(f"{out_dir}/edges.parquet")

train_parts = []
valid_parts = []
test_parts = []

neg_ratio = 10   # 1 positive : 10 negative
rng = np.random.default_rng(42)

for rg_idx in range(pf.num_row_groups):
    edges = pf.read_row_group(
        rg_idx,
        columns=["u", "v", "label_same_entity"]
    ).to_pandas()

    # мапим entity/split для u и v
    edges["entity_u"] = edges["u"].map(entity_map)
    edges["entity_v"] = edges["v"].map(entity_map)
    edges["split_u"] = edges["u"].map(split_map)
    edges["split_v"] = edges["v"].map(split_map)

    # берём только пары внутри одного split
    edges = edges[
        (edges["split_u"] == edges["split_v"]) &
        (edges["split_u"].notna())
    ].copy()

    if edges.empty:
        continue

    # positive + sampled negative
    pos = edges[edges["label_same_entity"] == 1].copy()
    neg = edges[edges["label_same_entity"] == 0].copy()

    if len(pos) > 0:
        neg_n = min(len(neg), len(pos) * neg_ratio)
        if neg_n > 0:
            neg_idx = rng.choice(neg.index.to_numpy(), size=neg_n, replace=False)
            neg = neg.loc[neg_idx]
        chunk = pd.concat([pos, neg], ignore_index=True)
    else:
        continue

    # джойн фичей по u/v
    u_feat = profiles.loc[chunk["u"]].reset_index(drop=True)
    v_feat = profiles.loc[chunk["v"]].reset_index(drop=True)

    feat = pd.DataFrame({
        "u": chunk["u"].values,
        "v": chunk["v"].values,
        "target": chunk["label_same_entity"].astype("int8").values,

        "same_email": safe_str_eq(u_feat["email"], v_feat["email"]),
        "same_phone": safe_str_eq(u_feat["phone"], v_feat["phone"]),
        "same_email_domain": safe_str_eq(u_feat["email_domain"], v_feat["email_domain"]),
        "same_phone_prefix": safe_str_eq(u_feat["phone_prefix"], v_feat["phone_prefix"]),
        "same_first_name": safe_str_eq(u_feat["first_name"], v_feat["first_name"]),
        "same_last_name": safe_str_eq(u_feat["last_name"], v_feat["last_name"]),

        "split_name": chunk["split_u"].values
    })

    # если эти колонки у тебя реально есть в profiles / source_profiles:
    optional_bool = [
        "sex", "country", "city_geoid", "fs_is_gmail",
        "fs_is_phone", "fs_is_yandex", "is_million_city"
    ]
    for col in optional_bool:
        if col in u_feat.columns and col in v_feat.columns:
            feat[f"same_{col}"] = safe_str_eq(
                u_feat[col].astype("string"),
                v_feat[col].astype("string")
            )

    optional_num = ["avg_tz_offset", "avg_local_hour", "visit_count"]
    for col in optional_num:
        if col in u_feat.columns and col in v_feat.columns:
            feat[f"abs_diff_{col}"] = safe_num_diff(u_feat[col], v_feat[col])

    optional_sets = ["device", "browser", "osfamily"]
    for col in optional_sets:
        if col in u_feat.columns and col in v_feat.columns:
            feat[f"jaccard_{col}"] = jaccard_series(u_feat[col], v_feat[col])

    split_name = chunk["split_u"].iloc[0]
    if split_name == "train":
        train_parts.append(feat)
    elif split_name == "valid":
        valid_parts.append(feat)
    elif split_name == "test":
        test_parts.append(feat)

In [28]:
# 5. Склеиваем
train_df = pd.concat(train_parts, ignore_index=True) if train_parts else pd.DataFrame()
valid_df = pd.concat(valid_parts, ignore_index=True) if valid_parts else pd.DataFrame()
test_df  = pd.concat(test_parts, ignore_index=True) if test_parts else pd.DataFrame()

print("train:", train_df.shape)
print("valid:", valid_df.shape)
print("test:", test_df.shape)

# -------------------------
# 6. Обучение CatBoost
# -------------------------
drop_cols = ["u", "v", "target", "split_name"]
feature_cols = [c for c in train_df.columns if c not in drop_cols]

X_train, y_train = train_df[feature_cols], train_df["target"]
X_valid, y_valid = valid_df[feature_cols], valid_df["target"]
X_test,  y_test  = test_df[feature_cols], test_df["target"]

model = CatBoostClassifier(
    iterations=500,
    depth=6,
    learning_rate=0.05,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=100
)

model.fit(
    X_train, y_train,
    eval_set=(X_valid, y_valid),
    use_best_model=True
)

# -------------------------
# 7. Оценка
# -------------------------
valid_pred_proba = model.predict_proba(X_valid)[:, 1]
test_pred_proba = model.predict_proba(X_test)[:, 1]

print("Valid ROC-AUC:", roc_auc_score(y_valid, valid_pred_proba))
print("Test ROC-AUC:", roc_auc_score(y_test, test_pred_proba))

test_pred = (test_pred_proba >= 0.5).astype(int)
print(classification_report(y_test, test_pred, digits=4))

# важность фич
fi = pd.DataFrame({
    "feature": feature_cols,
    "importance": model.get_feature_importance()
}).sort_values("importance", ascending=False)

print(fi.head(20))

train: (125169, 10)
valid: (28138, 10)
test: (26686, 10)
0:	test: 0.7343155	best: 0.7343155 (0)	total: 55ms	remaining: 27.4s
100:	test: 0.7330671	best: 0.7343876 (1)	total: 451ms	remaining: 1.78s
200:	test: 0.7330197	best: 0.7343876 (1)	total: 810ms	remaining: 1.2s
300:	test: 0.7330197	best: 0.7343876 (1)	total: 1.17s	remaining: 776ms
400:	test: 0.7330197	best: 0.7343876 (1)	total: 1.55s	remaining: 383ms
499:	test: 0.7330197	best: 0.7343876 (1)	total: 1.91s	remaining: 0us

bestTest = 0.7343875872
bestIteration = 1

Shrink model to first 2 iterations.
Valid ROC-AUC: 0.7343875871799277
Test ROC-AUC: 0.715897575659131
              precision    recall  f1-score   support

           0     0.9091    1.0000    0.9524     24260
           1     0.0000    0.0000    0.0000      2426

    accuracy                         0.9091     26686
   macro avg     0.4545    0.5000    0.4762     26686
weighted avg     0.8264    0.9091    0.8658     26686

             feature  importance
2  same_email_dom

/home/max/projects/mipt_case4/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/max/projects/mipt_case4/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/home/max/projects/mipt_case4/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.c